In [10]:
#!/usr/bin/env python
# coding: utf-8

# In[23]:


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns


# In[11]:


# I don't know if we can use the GPUs on DSMLP to utilize the CUDA function of Pytorch
# So do not set epoch too high in order to have a faster training process.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# In[12]:


# create varaibles that holds a dataframe
cwd = os.getcwd()
data_dir = os.path.join(cwd, "Data/")
data_files = [f for f in os.listdir(str(data_dir)) if f.endswith('csv')]

data_train_name = [f for f in data_files if 'TRAIN' in f]
data_test_name = [f for f in data_files if 'TEST' in f]
data_NPML_name = [f for f in data_files if 'NPML' in f]
data_NPML_cut_name = [f for f in data_files if 'npml_cut' in f]

train_df = pd.read_csv(os.path.join(data_dir,data_train_name[0]))
test_df = pd.read_csv(os.path.join(data_dir,data_test_name[0]))
NPML_df = pd.read_csv(os.path.join(data_dir,data_NPML_name[0]))

NPML_cut_id = np.array(pd.read_csv(os.path.join(data_dir,data_NPML_cut_name[0]))['id'])
NPML_cut_df = NPML_df[NPML_df['id'].isin(NPML_cut_id)]

train_df = train_df.dropna()
test_df = test_df.dropna()
NPML_df = NPML_df.dropna()
NPML_cut_df = NPML_cut_df.dropna()






Using device: cpu


In [11]:
NPML_df['predictions'] = np.load('predictions_NPML_original.npy')
NPML_df

,id,tdrift,tdrift50,tdrift10,rea,dcr,peakindex,peakvalue,tailslope,currentamp,lfpr,lq80,areagrowthrate,inflection point,risingedgeslope,predictions
0,3033789,82.917,41.5,8.3,-0.601381,3.543431e+05,1051,793.0,-0.068838,0.005179,0.018143,96140.0,-335964.5,312,11.135049,299.455750
1,3033790,179.820,90.0,18.0,-0.414141,2.496389e+05,1122,675.0,-0.052831,0.005456,0.012496,188266.0,-230906.0,292,4.204519,228.335754
2,3033791,184.815,92.5,18.5,-0.303937,2.738476e+05,1125,682.0,-0.057537,0.003668,0.014724,162153.0,-242375.0,312,4.401607,242.436981
3,3033792,200.799,100.5,20.1,0.069981,2.464699e+05,1137,709.0,-0.052356,0.003518,0.015758,223864.0,-225253.0,322,3.812199,225.550079
4,3033793,95.904,48.0,9.6,0.806482,3.080030e+05,1048,651.0,-0.057811,0.005508,0.017981,65933.0,-273412.0,324,7.679558,237.018250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159692,3193481,76.923,38.5,7.7,0.852405,8.809690e+05,1049,1928.0,-0.171740,0.005230,0.020454,145094.0,-858323.0,304,28.208502,736.519836
159693,3193482,93.906,47.0,9.4,1.409383,1.855922e+06,1061,4008.0,-0.369091,0.006652,0.020141,182525.0,-1830495.0,299,40.251158,1557.224243
159694,3193483,85.914,43.0,8.6,0.826311,1.400284e+06,1052,3009.0,-0.280138,0.007055,0.018795,137745.0,-1374344.0,299,42.189622,1185.412720
159695,3193484,64.935,32.5,6.5,-0.282476,2.521222e+06,1046,5338.0,-0.493204,0.006044,0.017436,328744.0,-2439512.0,339,107.706294,2096.330322


In [12]:
NPML_cut_df['predictions'] = np.load('predictions_NPML_cut_original.npy')
NPML_cut_df

,id,tdrift,tdrift50,tdrift10,rea,dcr,peakindex,peakvalue,tailslope,currentamp,lfpr,lq80,areagrowthrate,inflection point,risingedgeslope,predictions
4,3033793,95.904,48.0,9.6,0.806482,3.080030e+05,1048,651.0,-0.057811,0.005508,0.017981,65933.0,-273412.0,324,7.679558,237.018250
5,3033794,123.876,62.0,12.4,1.648333,1.735847e+06,1090,4197.0,-0.390518,0.007440,0.020745,231249.0,-1897374.0,315,29.034052,1635.438965
6,3033795,99.900,50.0,10.0,1.491345,3.559872e+05,1050,844.0,-0.068931,0.005345,0.023162,113615.0,-341986.0,327,6.909205,296.822815
7,3033796,124.875,62.5,12.5,1.418168,2.952572e+05,1054,686.0,-0.054620,0.005573,0.021739,107245.0,-262569.0,382,4.558255,235.864532
8,3033797,114.885,57.5,11.5,1.066913,2.650152e+05,1049,743.0,-0.053453,0.005072,0.017822,191027.0,-252230.0,328,5.511560,230.530029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159681,3193470,91.908,46.0,9.2,1.022303,1.151945e+06,1058,2557.0,-0.231814,0.004541,0.020273,145287.0,-1150052.0,314,29.826326,977.282593
159682,3193471,125.874,63.0,12.6,0.475891,2.928008e+05,1051,685.0,-0.055601,0.006111,0.020711,127036.0,-253489.0,335,6.398803,235.545898
159685,3193474,76.923,38.5,7.7,0.648722,6.382518e+05,1040,1402.0,-0.122663,0.006581,0.018843,129230.0,-607133.0,339,21.903544,526.249451
159691,3193480,84.915,42.5,8.5,0.646862,8.554475e+05,1050,1868.0,-0.169065,0.006413,0.019633,146799.0,-823513.0,310,27.041196,714.895203


In [13]:
NPML_cut_final = NPML_cut_df[['id','predictions']]
NPML_final = NPML_df[['id','predictions']]

In [14]:
NPML_final.to_csv('NPML_final.csv',index=False)
NPML_cut_final.to_csv('NPML_cut_final.csv',index=False)

In [17]:
NPML_cut_final.head(20)

,id,predictions
4,3033793,234.967590
5,3033794,1683.810669
6,3033795,299.244110
7,3033796,235.379486
8,3033797,227.831238
12,3033801,2596.727295
13,3033802,708.671143
16,3033805,255.109741
18,3033807,294.106567
19,3033808,2098.194580


In [17]:
abc = pd.read_csv(os.path.join(data_dir,data_NPML_name[0]))
abc[abc.isnull().any(axis=1)]

,id,tdrift,tdrift50,tdrift10,rea,dcr,peakindex,peakvalue,tailslope,currentamp,lfpr,lq80,areagrowthrate,inflection point,risingedgeslope
116829,3150618,-707.292,-354.0,-70.8,NaN,81106.21875,178,110.0,-0.001382,0.015947,0.0,0.0,0.0,0,NaN


In [20]:
abc[abc['id'] == 31506181]

,id,tdrift,tdrift50,tdrift10,rea,dcr,peakindex,peakvalue,tailslope,currentamp,lfpr,lq80,areagrowthrate,inflection point,risingedgeslope
